# v04 3클래스 리텐션 모델링

- 입력: v04 코호트 기준 Core 43 피처
- 5-Fold 확장형 시간 검증에서 모델·임계값 선택
- 최종 학습: 선정연도 2010~2017
- 최종 Test: 선정연도 2018 → 타깃연도 2019
- Test 결과는 모델 조건 선택에 사용하지 않음
- 클래스 점수는 보정 확률이 아니라 순위·판단용 모델 점수

In [1]:
from __future__ import annotations

import hashlib
import itertools
import json
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

SEED = 42
CLASS_CODES = [0, 1, 2]
CLASS_NAMES = ['retained', 'weakened', 'stopped']
CLASS_LABELS_KO = {0: '파워 지위 유지', 1: '파워 지위 약화', 2: '리뷰 활동 중단'}
PRIMARY_TARGET_RATE = 0.20
TOP_K_RATES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
TIME_FOLDS = [
    (2012, 2013),
    (2013, 2014),
    (2014, 2015),
    (2015, 2016),
    (2016, 2017),
]
PENALTIES = ['l1', 'l2']
C_VALUES = [0.01, 0.03, 0.10, 0.30, 1.00]
CLASS_WEIGHT_OPTIONS = {'none': None, 'balanced': 'balanced'}
WEAKENED_THRESHOLDS = [0.30, 0.36, 0.42]
STOPPED_THRESHOLDS = [0.35, 0.45, 0.55]

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODELING_PATH = PROJECT_ROOT / 'data' / 'processed' / 'modeling_dataset_rolling_v04.parquet'
V02_METADATA_PATH = PROJECT_ROOT / 'models' / 'final_core_hgb_metadata_v02.json'
V03_METADATA_PATH = PROJECT_ROOT / 'models' / 'final_core_logistic_multiclass_metadata_v03.json'
MODEL_PATH = PROJECT_ROOT / 'models' / 'final_core_logistic_multiclass_v04.joblib'
METADATA_PATH = PROJECT_ROOT / 'models' / 'final_core_logistic_multiclass_metadata_v04.json'
PROFILE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'predictions' / 'final_test_retention_profiles_v04.parquet'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'modeling' / 'multiclass_model_performance_v04.md'
VALIDATION_PATH = PROJECT_ROOT / 'reports' / 'tables' / 'multiclass_validation_results_v04.csv'
CANDIDATE_PATH = PROJECT_ROOT / 'reports' / 'tables' / 'multiclass_model_candidates_v04.csv'
CONFUSION_PATH = PROJECT_ROOT / 'reports' / 'tables' / 'multiclass_confusion_matrix_v04.csv'
TOP_K_PATH = PROJECT_ROOT / 'reports' / 'tables' / 'multiclass_top_k_performance_v04.csv'

for directory in [MODEL_PATH.parent, PROFILE_PATH.parent, REPORT_PATH.parent, VALIDATION_PATH.parent]:
    directory.mkdir(parents=True, exist_ok=True)

feature_columns = json.loads(V02_METADATA_PATH.read_text(encoding='utf-8'))['feature_columns']
frame = pd.read_parquet(MODELING_PATH)
assert len(feature_columns) == 43
assert len(frame) == 37_953
assert frame['sample_id'].is_unique
assert not (set(feature_columns) - set(frame.columns))
assert not ({'target_review_count', 'target_active_months', 'retention_state', 'churn'} & set(feature_columns))
assert np.isinf(frame[feature_columns].to_numpy(dtype=float)).sum() == 0
assert frame.loc[frame['selection_year'].eq(2018), 'retention_state'].value_counts().to_dict() == {1: 3_065, 0: 2_584, 2: 884}

def build_model(penalty: str, c_value: float, class_weight) -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            C=c_value,
            penalty=penalty,
            solver='saga',
            class_weight=class_weight,
            max_iter=5_000,
            tol=1e-3,
            random_state=SEED,
        )),
    ])

def threshold_predictions(scores: np.ndarray, weakened_threshold: float, stopped_threshold: float) -> np.ndarray:
    predictions = np.zeros(len(scores), dtype='int8')
    predictions[scores[:, 1] >= weakened_threshold] = 1
    predictions[scores[:, 2] >= stopped_threshold] = 2
    return predictions

def evaluate(y_true: np.ndarray, predictions: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, predictions, labels=CLASS_CODES, zero_division=0
    )
    y_binary = label_binarize(y_true, classes=CLASS_CODES)
    result = {
        'accuracy': accuracy_score(y_true, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_true, predictions),
        'macro_precision': float(precision.mean()),
        'macro_recall': float(recall.mean()),
        'macro_f1': f1_score(y_true, predictions, average='macro'),
        'weighted_f1': f1_score(y_true, predictions, average='weighted'),
        'macro_pr_auc': float(np.mean([
            average_precision_score(y_binary[:, index], scores[:, index]) for index in CLASS_CODES
        ])),
        'macro_ovr_roc_auc': roc_auc_score(y_true, scores, multi_class='ovr', average='macro'),
    }
    for index, class_name in enumerate(CLASS_NAMES):
        result[f'{class_name}_precision'] = precision[index]
        result[f'{class_name}_recall'] = recall[index]
        result[f'{class_name}_f1'] = f1[index]
        result[f'{class_name}_support'] = int(support[index])
        result[f'{class_name}_pr_auc'] = average_precision_score(y_binary[:, index], scores[:, index])
        result[f'{class_name}_roc_auc'] = roc_auc_score(y_binary[:, index], scores[:, index])
    return result

def confusion_records(split: str, y_true: np.ndarray, predictions: np.ndarray) -> list[dict]:
    matrix = confusion_matrix(y_true, predictions, labels=CLASS_CODES)
    return [
        {
            'split': split,
            'actual_state': CLASS_NAMES[actual],
            'predicted_state': CLASS_NAMES[predicted],
            'users': int(matrix[actual, predicted]),
        }
        for actual in CLASS_CODES for predicted in CLASS_CODES
    ]

def top_k_records(split: str, y_true: np.ndarray, scores: np.ndarray) -> list[dict]:
    rankings = {
        'unified': scores[:, 1] + scores[:, 2],
        'stopped_only': scores[:, 2],
        'weakened_only': scores[:, 1],
    }
    status_loss = y_true != 0
    stopped = y_true == 2
    weakened = y_true == 1
    records = []
    for ranking_name, ranking_score in rankings.items():
        order = np.argsort(-ranking_score, kind='stable')
        for rate in TOP_K_RATES:
            users = int(np.ceil(len(y_true) * rate))
            selected = order[:users]
            captured_status = int(status_loss[selected].sum())
            captured_stopped = int(stopped[selected].sum())
            captured_weakened = int(weakened[selected].sum())
            precision = captured_status / users
            records.append({
                'split': split,
                'ranking': ranking_name,
                'target_rate': rate,
                'target_users': users,
                'status_loss_captured': captured_status,
                'status_loss_precision': precision,
                'status_loss_recall': captured_status / status_loss.sum(),
                'status_loss_lift': precision / status_loss.mean(),
                'stopped_captured': captured_stopped,
                'stopped_recall': captured_stopped / stopped.sum(),
                'weakened_captured': captured_weakened,
                'weakened_recall': captured_weakened / weakened.sum(),
            })
    return records

display(frame.groupby(['selection_year', 'retention_state']).size().unstack(fill_value=0))


retention_state,0,1,2
selection_year,,,
2010,660,657,251
2011,966,1074,463
2012,1220,1092,441
2013,1435,1485,528
2014,1772,1884,636
2015,1987,2538,890
2016,2147,2528,873
2017,2476,2624,793
2018,2584,3065,884


In [2]:
warnings.filterwarnings('ignore', category=ConvergenceWarning)
candidate_rows = []

model_specs = list(itertools.product(PENALTIES, C_VALUES, CLASS_WEIGHT_OPTIONS.items()))
for penalty, c_value, (class_weight_name, class_weight) in model_specs:
    oof_parts = []
    fold_pr_auc = []
    fold_roc_auc = []
    convergence_iterations = []
    candidate_id = f'{penalty}_C{c_value:g}_{class_weight_name}'

    for train_end, validation_year in TIME_FOLDS:
        train_fold = frame[frame['selection_year'].between(2010, train_end)]
        validation_fold = frame[frame['selection_year'].eq(validation_year)]
        model = build_model(penalty, c_value, class_weight)
        model.fit(train_fold[feature_columns], train_fold['retention_state'])
        scores = model.predict_proba(validation_fold[feature_columns])
        argmax_predictions = scores.argmax(axis=1).astype('int8')
        base_metrics = evaluate(validation_fold['retention_state'].to_numpy(), argmax_predictions, scores)
        fold_pr_auc.append(base_metrics['macro_pr_auc'])
        fold_roc_auc.append(base_metrics['macro_ovr_roc_auc'])
        convergence_iterations.append(int(model.named_steps['model'].n_iter_.max()))
        part = validation_fold[['sample_id', 'selection_year', 'retention_state']].copy()
        part[['retained_score', 'weakened_score', 'stopped_score']] = scores
        oof_parts.append(part)

    oof = pd.concat(oof_parts, ignore_index=True)
    oof_scores = oof[['retained_score', 'weakened_score', 'stopped_score']].to_numpy()
    oof_y = oof['retention_state'].to_numpy()
    for weakened_threshold, stopped_threshold in itertools.product(WEAKENED_THRESHOLDS, STOPPED_THRESHOLDS):
        predictions = threshold_predictions(oof_scores, weakened_threshold, stopped_threshold)
        metrics = evaluate(oof_y, predictions, oof_scores)
        candidate_rows.append({
            'candidate_id': candidate_id,
            'penalty': penalty,
            'C': c_value,
            'class_weight': class_weight_name,
            'weakened_threshold': weakened_threshold,
            'stopped_threshold': stopped_threshold,
            'validation_samples': len(oof),
            'fold_macro_pr_auc_mean': float(np.mean(fold_pr_auc)),
            'fold_macro_pr_auc_std': float(np.std(fold_pr_auc, ddof=1)),
            'fold_macro_roc_auc_mean': float(np.mean(fold_roc_auc)),
            'maximum_iterations': max(convergence_iterations),
            **{f'oof_{key}': value for key, value in metrics.items()},
        })

candidate_df = pd.DataFrame(candidate_rows)
candidate_df = candidate_df.sort_values(
    ['oof_macro_f1', 'oof_macro_pr_auc', 'oof_balanced_accuracy', 'oof_stopped_recall', 'oof_weakened_recall'],
    ascending=False,
    kind='stable',
).reset_index(drop=True)
candidate_df.insert(0, 'selection_rank', np.arange(1, len(candidate_df) + 1))
candidate_df['selected'] = candidate_df['selection_rank'].eq(1)
candidate_df.to_csv(CANDIDATE_PATH, index=False, encoding='utf-8-sig')

selected = candidate_df.iloc[0]
selected_penalty = str(selected['penalty'])
selected_c = float(selected['C'])
selected_class_weight_name = str(selected['class_weight'])
selected_class_weight = CLASS_WEIGHT_OPTIONS[selected_class_weight_name]
selected_weakened_threshold = float(selected['weakened_threshold'])
selected_stopped_threshold = float(selected['stopped_threshold'])

print('selected model:', selected['candidate_id'])
print('selected thresholds:', selected_weakened_threshold, selected_stopped_threshold)
display(candidate_df.head(10))


selected model: l1_C0.01_balanced
selected thresholds: 0.36 0.45


,selection_rank,candidate_id,penalty,C,class_weight,weakened_threshold,stopped_threshold,validation_samples,fold_macro_pr_auc_mean,fold_macro_pr_auc_std,...,oof_weakened_support,oof_weakened_pr_auc,oof_weakened_roc_auc,oof_stopped_precision,oof_stopped_recall,oof_stopped_f1,oof_stopped_support,oof_stopped_pr_auc,oof_stopped_roc_auc,selected
0,1,l1_C0.01_balanced,l1,0.01,balanced,0.36,0.45,24596,0.579550,0.008100,...,11059,0.588579,0.671613,0.406442,0.532527,0.461019,3720,0.410704,0.804334,True
1,2,l1_C0.1_none,l1,0.10,none,0.42,0.35,24596,0.582145,0.006076,...,11059,0.592692,0.680863,0.472583,0.352151,0.403574,3720,0.411903,0.805190,False
2,3,l1_C0.03_balanced,l1,0.03,balanced,0.36,0.45,24596,0.580784,0.006693,...,11059,0.589059,0.671714,0.403368,0.540860,0.462104,3720,0.412223,0.805158,False
3,4,l2_C0.1_none,l2,0.10,none,0.42,0.35,24596,0.582272,0.005769,...,11059,0.593139,0.680805,0.470799,0.353226,0.403625,3720,0.411393,0.805018,False
4,5,l2_C0.03_none,l2,0.03,none,0.42,0.35,24596,0.582250,0.005863,...,11059,0.593100,0.680833,0.472655,0.350806,0.402716,3720,0.411283,0.805010,False
5,6,l1_C0.3_none,l1,0.30,none,0.42,0.35,24596,0.582413,0.005719,...,11059,0.593038,0.680886,0.469936,0.352957,0.403132,3720,0.411812,0.805249,False
6,7,l2_C0.3_none,l2,0.30,none,0.42,0.35,24596,0.582263,0.005691,...,11059,0.593052,0.680757,0.470903,0.354570,0.404539,3720,0.411366,0.804928,False
7,8,l2_C0.01_balanced,l2,0.01,balanced,0.36,0.45,24596,0.580491,0.006868,...,11059,0.588765,0.671030,0.403509,0.544086,0.463370,3720,0.412086,0.804909,False
8,9,l2_C0.03_balanced,l2,0.03,balanced,0.36,0.45,24596,0.581005,0.006510,...,11059,0.589222,0.671506,0.402372,0.547312,0.463781,3720,0.412301,0.805056,False
9,10,l2_C1_none,l2,1.00,none,0.42,0.35,24596,0.582215,0.005715,...,11059,0.593015,0.680707,0.470714,0.354301,0.404294,3720,0.411363,0.804828,False


In [3]:
validation_records = []
confusion_rows = []
selected_oof_parts = []

for fold_number, (train_end, validation_year) in enumerate(TIME_FOLDS, start=1):
    train_fold = frame[frame['selection_year'].between(2010, train_end)]
    validation_fold = frame[frame['selection_year'].eq(validation_year)]
    model = build_model(selected_penalty, selected_c, selected_class_weight)
    model.fit(train_fold[feature_columns], train_fold['retention_state'])
    scores = model.predict_proba(validation_fold[feature_columns])
    predictions = threshold_predictions(scores, selected_weakened_threshold, selected_stopped_threshold)
    validation_records.append({
        'record_type': 'fold',
        'split': f'fold_{fold_number}',
        'train_selection_years': f'2010~{train_end}',
        'validation_selection_year': validation_year,
        'train_samples': len(train_fold),
        'validation_samples': len(validation_fold),
        **evaluate(validation_fold['retention_state'].to_numpy(), predictions, scores),
    })
    confusion_rows.extend(confusion_records(
        f'validation_{validation_year}', validation_fold['retention_state'].to_numpy(), predictions
    ))
    part = validation_fold[['sample_id', 'selection_year', 'retention_state']].copy()
    part[['retained_score', 'weakened_score', 'stopped_score']] = scores
    selected_oof_parts.append(part)

fold_df = pd.DataFrame(validation_records)
metric_columns = [column for column in fold_df.columns if column not in {
    'record_type', 'split', 'train_selection_years', 'validation_selection_year',
    'train_samples', 'validation_samples',
}]
summary_rows = []
for statistic in ['mean', 'std']:
    row = {
        'record_type': statistic,
        'split': 'time_5_fold',
        'train_selection_years': 'expanding_2010~2016',
        'validation_selection_year': pd.NA,
        'train_samples': pd.NA,
        'validation_samples': pd.NA,
    }
    for column in metric_columns:
        row[column] = fold_df[column].mean() if statistic == 'mean' else fold_df[column].std(ddof=1)
    summary_rows.append(row)

oof_df = pd.concat(selected_oof_parts, ignore_index=True)
oof_scores = oof_df[['retained_score', 'weakened_score', 'stopped_score']].to_numpy()
oof_y = oof_df['retention_state'].to_numpy()
oof_predictions = threshold_predictions(oof_scores, selected_weakened_threshold, selected_stopped_threshold)
oof_record = {
    'record_type': 'pooled_oof',
    'split': 'time_5_fold',
    'train_selection_years': 'expanding_2010~2016',
    'validation_selection_year': pd.NA,
    'train_samples': pd.NA,
    'validation_samples': len(oof_df),
    **evaluate(oof_y, oof_predictions, oof_scores),
}

final_train_df = frame[frame['selection_year'].between(2010, 2017)].copy()
final_test_df = frame[frame['selection_year'].eq(2018)].copy()
assert len(final_train_df) == 31_420
assert len(final_test_df) == 6_533
assert final_train_df['target_year'].max() == 2018
assert final_test_df['target_year'].eq(2019).all()

final_model = build_model(selected_penalty, selected_c, selected_class_weight)
final_model.fit(final_train_df[feature_columns], final_train_df['retention_state'])
test_scores = final_model.predict_proba(final_test_df[feature_columns])
test_predictions = threshold_predictions(test_scores, selected_weakened_threshold, selected_stopped_threshold)
test_y = final_test_df['retention_state'].to_numpy()
test_metrics = evaluate(test_y, test_predictions, test_scores)
test_record = {
    'record_type': 'final_test',
    'split': 'selection_2018_target_2019',
    'train_selection_years': '2010~2017',
    'validation_selection_year': 2018,
    'train_samples': len(final_train_df),
    'validation_samples': len(final_test_df),
    **test_metrics,
}

validation_df = pd.concat([
    fold_df,
    pd.DataFrame(summary_rows),
    pd.DataFrame([oof_record, test_record]),
], ignore_index=True)
validation_df.to_csv(VALIDATION_PATH, index=False, encoding='utf-8-sig')

confusion_rows.extend(confusion_records('pooled_oof', oof_y, oof_predictions))
confusion_rows.extend(confusion_records('final_test', test_y, test_predictions))
confusion_df = pd.DataFrame(confusion_rows)
confusion_df.to_csv(CONFUSION_PATH, index=False, encoding='utf-8-sig')

top_k_df = pd.DataFrame(
    top_k_records('pooled_oof', oof_y, oof_scores)
    + top_k_records('final_test', test_y, test_scores)
)
top_k_df.to_csv(TOP_K_PATH, index=False, encoding='utf-8-sig')

profile_df = final_test_df.copy()
profile_df['retention_state_label'] = profile_df['retention_state'].map(CLASS_LABELS_KO)
profile_df[['retained_score', 'weakened_score', 'stopped_score']] = test_scores
profile_df['priority_score'] = profile_df['weakened_score'] + profile_df['stopped_score']
profile_df['predicted_state'] = test_predictions
profile_df['predicted_state_label'] = profile_df['predicted_state'].map(CLASS_LABELS_KO)
profile_df['priority_rank'] = profile_df['priority_score'].rank(method='first', ascending=False).astype(int)
profile_df['priority_top_percent'] = profile_df['priority_rank'] / len(profile_df) * 100
target_users = int(np.ceil(len(profile_df) * PRIMARY_TARGET_RATE))
profile_df['selected_for_crm'] = profile_df['priority_rank'].le(target_users).astype('int8')
profile_df = profile_df.sort_values(['priority_rank', 'sample_id']).reset_index(drop=True)
profile_df.to_parquet(PROFILE_PATH, index=False)

joblib.dump(final_model, MODEL_PATH)
reloaded_model = joblib.load(MODEL_PATH)
assert np.allclose(test_scores, reloaded_model.predict_proba(final_test_df[feature_columns]), rtol=0, atol=1e-12)
model_checksum = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()

top20 = top_k_df[
    top_k_df['split'].eq('final_test')
    & top_k_df['ranking'].eq('unified')
    & top_k_df['target_rate'].eq(PRIMARY_TARGET_RATE)
].iloc[0]

subgroup_metrics = {}
for group_value, group_name in [(0, 'no_prior_activity'), (1, 'has_prior_activity')]:
    mask = final_test_df['prior_activity_available'].eq(group_value).to_numpy()
    subgroup_metrics[group_name] = evaluate(test_y[mask], test_predictions[mask], test_scores[mask])

v03_metadata = json.loads(V03_METADATA_PATH.read_text(encoding='utf-8'))
metadata = {
    'version': 'v04',
    'model_name': 'Core Multiclass Logistic v04',
    'model_type': 'LogisticRegression',
    'problem_type': 'multiclass_classification',
    'class_map': {str(code): name for code, name in zip(CLASS_CODES, CLASS_NAMES)},
    'class_labels_ko': {str(key): value for key, value in CLASS_LABELS_KO.items()},
    'label_definition': {
        'retained': 'target_review_count >= 10 and target_active_months >= 3',
        'weakened': 'target_review_count >= 1 and (target_review_count < 10 or target_active_months < 3)',
        'stopped': 'target_review_count == 0',
    },
    'cohort_definition': 'selection year review_count >= 10 and active_months >= 3; no H2 continuity filter',
    'time_structure': 'comparison Y-1, selection/feature cutoff Y, target Y+1',
    'selection_rule': 'highest pooled 5-fold macro F1; ties by PR-AUC, balanced accuracy and class recall',
    'model_parameters': {
        'penalty': selected_penalty,
        'C': selected_c,
        'solver': 'saga',
        'class_weight': selected_class_weight_name,
        'max_iter': 5000,
        'tol': 0.001,
        'random_state': SEED,
    },
    'decision_thresholds': {
        'weakened_score': selected_weakened_threshold,
        'stopped_score': selected_stopped_threshold,
        'evaluation_order': ['stopped', 'weakened', 'retained'],
    },
    'priority_policy': {
        'score': 'weakened_score + stopped_score',
        'primary_target_rate': PRIMARY_TARGET_RATE,
    },
    'feature_set': 'activity+interval+business',
    'feature_count': len(feature_columns),
    'feature_columns': feature_columns,
    'imputed_feature_count': len(final_model.named_steps['imputer'].get_feature_names_out(feature_columns)),
    'time_folds': [
        {'train_selection_years': f'2010~{train_end}', 'validation_selection_year': validation_year}
        for train_end, validation_year in TIME_FOLDS
    ],
    'final_train_selection_years': '2010~2017',
    'test_selection_year': 2018,
    'test_target_year': 2019,
    'test_samples': len(final_test_df),
    'test_metrics': {key: float(value) for key, value in test_metrics.items()},
    'test_subgroup_metrics': {
        group: {key: float(value) for key, value in metrics.items()}
        for group, metrics in subgroup_metrics.items()
    },
    'top20_policy': {key: float(top20[key]) for key in [
        'target_users', 'status_loss_captured', 'status_loss_precision', 'status_loss_recall',
        'status_loss_lift', 'stopped_captured', 'stopped_recall', 'weakened_captured', 'weakened_recall',
    ]},
    'v03_reference_metrics': v03_metadata['test_metrics'],
    'model_sha256': model_checksum,
    'python_version': sys.version.split()[0],
    'sklearn_version': sklearn.__version__,
    'pandas_version': pd.__version__,
    'score_warning': '클래스별 점수는 확률 보정 전 모델 점수이며 실제 상태 확률로 표현하지 않는다.',
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


8612

In [4]:
v03_metrics = v03_metadata['test_metrics']
report = f'''# 파워 리뷰어 3클래스 리텐션 모델 v04

## 모델 선정 원칙

- 2013~2017 확장형 시간 5-Fold에서만 규제·가중치·임계값 선택
- 선정 기준: pooled OOF Macro F1 우선, 이후 PR-AUC·Balanced Accuracy·클래스 Recall
- 최종 Train: 선정연도 2010~2017, 31,420표본
- 최종 Test: 선정연도 2018 → 타깃연도 2019, 6,533표본
- Test 결과는 모델 조건 선택에 사용하지 않음

## 선택된 조건

- Penalty: {selected_penalty}
- C: {selected_c:g}
- Class weight: {selected_class_weight_name}
- 약화 임계값: {selected_weakened_threshold:.2f}
- 중단 임계값: {selected_stopped_threshold:.2f}

## 최종 Test 성능

| 지표 | v03 참고 | v04 |
|---|---:|---:|
| Macro F1 | {v03_metrics['macro_f1']:.4f} | {test_metrics['macro_f1']:.4f} |
| Macro PR-AUC | {v03_metrics['macro_pr_auc']:.4f} | {test_metrics['macro_pr_auc']:.4f} |
| Macro ROC-AUC | {v03_metrics['macro_ovr_roc_auc']:.4f} | {test_metrics['macro_ovr_roc_auc']:.4f} |
| Balanced Accuracy | {v03_metrics['balanced_accuracy']:.2%} | {test_metrics['balanced_accuracy']:.2%} |

| 클래스 | Precision | Recall | F1 | PR-AUC |
|---|---:|---:|---:|---:|
| 유지 | {test_metrics['retained_precision']:.2%} | {test_metrics['retained_recall']:.2%} | {test_metrics['retained_f1']:.4f} | {test_metrics['retained_pr_auc']:.4f} |
| 약화 | {test_metrics['weakened_precision']:.2%} | {test_metrics['weakened_recall']:.2%} | {test_metrics['weakened_f1']:.4f} | {test_metrics['weakened_pr_auc']:.4f} |
| 중단 | {test_metrics['stopped_precision']:.2%} | {test_metrics['stopped_recall']:.2%} | {test_metrics['stopped_f1']:.4f} | {test_metrics['stopped_pr_auc']:.4f} |

v03과 v04는 코호트와 시간 구조가 다르므로 성능 수치는 단순 우열이 아니라 참고 비교로만 사용한다.

## 통합 상위 20% 정책

- 관리 대상: {int(top20['target_users']):,}명
- 실제 지위 상실 포착: {int(top20['status_loss_captured']):,}명
- 지위 상실 Precision: {top20['status_loss_precision']:.2%}
- 지위 상실 Recall: {top20['status_loss_recall']:.2%}
- 무작위 대비 Lift: {top20['status_loss_lift']:.2f}배
- 중단 포착: {int(top20['stopped_captured']):,}명, Recall {top20['stopped_recall']:.2%}
- 약화 포착: {int(top20['weakened_captured']):,}명, Recall {top20['weakened_recall']:.2%}

전체 지위 상실자가 3,949명이므로 상위 20% 정책의 Recall 이론적 최대치는 약 33.10%다.

## 제한사항

- 클래스 점수는 보정된 실제 확률이 아니다.
- 2017년 활동이 없는 후보는 결측 표시를 통해 학습하며, 초기 연도 데이터 희소성 영향이 남을 수 있다.
- CRM 개입 효과와 복귀 결과는 현재 데이터에 없다.
- v04 모델의 운영 기본값 전환은 별도 승인 후 수행한다.
'''
REPORT_PATH.write_text(report, encoding='utf-8')

expected_paths = [
    MODEL_PATH, METADATA_PATH, PROFILE_PATH, REPORT_PATH,
    VALIDATION_PATH, CANDIDATE_PATH, CONFUSION_PATH, TOP_K_PATH,
]
assert all(path.exists() and path.stat().st_size > 0 for path in expected_paths)
assert len(profile_df) == 6_533
assert int(profile_df['selected_for_crm'].sum()) == 1_307
assert len(candidate_df) == len(PENALTIES) * len(C_VALUES) * len(CLASS_WEIGHT_OPTIONS) * len(WEAKENED_THRESHOLDS) * len(STOPPED_THRESHOLDS)

print('v04 model artifacts created')
print('selected:', selected['candidate_id'], selected_weakened_threshold, selected_stopped_threshold)
print('pooled OOF Macro F1:', round(float(selected['oof_macro_f1']), 6))
print('final Test Macro F1:', round(test_metrics['macro_f1'], 6))
print('final Test Macro PR-AUC:', round(test_metrics['macro_pr_auc'], 6))
print('model SHA256:', model_checksum)
display(validation_df.tail(4))
display(top_k_df[(top_k_df['split'].eq('final_test')) & (top_k_df['ranking'].eq('unified'))])


v04 model artifacts created
selected: l1_C0.01_balanced 0.36 0.45
pooled OOF Macro F1: 0.556162
final Test Macro F1: 0.552098
final Test Macro PR-AUC: 0.579236
model SHA256: a737d29fcf2b99e291797501e5ac7ca62a8341e32b935f64f24eb14c1e801e09


,record_type,split,train_selection_years,validation_selection_year,train_samples,validation_samples,accuracy,balanced_accuracy,macro_precision,macro_recall,...,weakened_f1,weakened_support,weakened_pr_auc,weakened_roc_auc,stopped_precision,stopped_recall,stopped_f1,stopped_support,stopped_pr_auc,stopped_roc_auc
5,mean,time_5_fold,expanding_2010~2016,<NA>,<NA>,<NA>,0.575652,0.566020,0.551591,0.566020,...,0.567348,2211.80000,0.586209,0.670338,0.405094,0.529025,0.458741,744.000000,0.410162,0.802713
6,std,time_5_fold,expanding_2010~2016,<NA>,<NA>,<NA>,0.005737,0.007989,0.006489,0.007989,...,0.014934,502.98827,0.017463,0.009049,0.014309,0.021830,0.015844,157.065273,0.015207,0.011054
7,pooled_oof,time_5_fold,expanding_2010~2016,<NA>,<NA>,24596,0.576191,0.567354,0.552490,0.567354,...,0.569690,11059.00000,0.588579,0.671613,0.406442,0.532527,0.461019,3720.000000,0.410704,0.804334
8,final_test,selection_2018_target_2019,2010~2017,2018,31420,6533,0.577529,0.564859,0.546117,0.564859,...,0.586269,3065.00000,0.611288,0.674106,0.396389,0.521493,0.450415,884.000000,0.408924,0.816883


,split,ranking,target_rate,target_users,status_loss_captured,status_loss_precision,status_loss_recall,status_loss_lift,stopped_captured,stopped_recall,weakened_captured,weakened_recall
24,final_test,unified,0.05,327,308,0.941896,0.077994,1.558219,178,0.201357,130,0.042414
25,final_test,unified,0.10,654,600,0.917431,0.151937,1.517746,297,0.335973,303,0.098858
26,final_test,unified,0.15,980,867,0.884694,0.219549,1.463587,388,0.438914,479,0.156281
27,final_test,unified,0.20,1307,1142,0.873757,0.289187,1.445493,481,0.544118,661,0.215661
28,final_test,unified,0.25,1634,1409,0.862301,0.356799,1.426542,551,0.623303,858,0.279935
29,final_test,unified,0.30,1960,1659,0.846429,0.420106,1.400283,605,0.684389,1054,0.343883
30,final_test,unified,0.35,2287,1907,0.833843,0.482907,1.379463,655,0.740950,1252,0.408483
31,final_test,unified,0.40,2614,2153,0.823642,0.545201,1.362586,704,0.796380,1449,0.472757
